<a href="https://colab.research.google.com/github/alokkumary0715-lgtm/Rag_pipeline/blob/main/Rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q fastapi python-multipart httpx fpdf

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [13]:
from fpdf import FPDF


with open("hr_policy.txt", "w") as f:
    f.write("""COMPANY HR POLICY 2026
1. Annual Leave: Employees are allowed 20 paid leaves per calendar year.
2. Work From Home: Hybrid work allows up to 2 WFH days per week with manager approval.
3. Working Hours: Core hours are 10:00 AM to 6:00 PM IST.
""")


pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial", size=12)
pdf.cell(200, 10, txt="Employee Handbook - IT Security Guidelines", ln=True)
pdf.multi_cell(0, 10, txt="All employees must use VPN when accessing internal databases. Passwords must be rotated every 90 days.")
pdf.output("employee_handbook.pdf")


with open("salary_data.csv", "w") as f:
    f.write("emp_id,name,salary\n101,Amit,85000\n102,Neha,92000")


with open("large_file.txt", "wb") as f:
    f.write(b"0" * (11 * 1024 * 1024))

print("Test files ban gayi hain: hr_policy.txt, employee_handbook.pdf, salary_data.csv, large_file.txt")

Test files ban gayi hain: hr_policy.txt, employee_handbook.pdf, salary_data.csv, large_file.txt


In [14]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.testclient import TestClient
import shutil
import os
from pathlib import Path

app = FastAPI()

UPLOAD_DIR = Path("uploaded_documents")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

ALLOWED_EXTENSIONS = {".pdf", ".docx", ".txt"}
MAX_FILE_SIZE = 10 * 1024 * 1024  # 10 MB

@app.post("/documents/upload")
async def upload_document(file: UploadFile = File(...)):
    file_ext = os.path.splitext(file.filename)[1].lower()
    if file_ext not in ALLOWED_EXTENSIONS:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid file type. Allowed types: {', '.join(ALLOWED_EXTENSIONS)}"
        )

    file.file.seek(0, 2)
    file_size = file.file.tell()
    file.file.seek(0)

    if file_size > MAX_FILE_SIZE:
        raise HTTPException(
            status_code=400,
            detail="File size exceeds the 10 MB limit."
        )

    file_path = UPLOAD_DIR / file.filename
    try:
        with open(file_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"File save karne mein error: {str(e)}")
    finally:
        await file.close()

    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "size_bytes": file_size,
        "status": "Uploaded successfully"
    }


client = TestClient(app)

# Test Case 1: Valid PDF upload
with open("employee_handbook.pdf", "rb") as f:
    res1 = client.post("/documents/upload", files={"file": ("employee_handbook.pdf", f, "application/pdf")})
print("Test 1 (Valid PDF):", res1.status_code, res1.json())

# Test Case 2: Valid TXT upload
with open("hr_policy.txt", "rb") as f:
    res2 = client.post("/documents/upload", files={"file": ("hr_policy.txt", f, "text/plain")})
print("Test 2 (Valid TXT):", res2.status_code, res2.json())

# Test Case 3: Invalid file type (.csv)
with open("salary_data.csv", "rb") as f:
    res3 = client.post("/documents/upload", files={"file": ("salary_data.csv", f, "text/csv")})
print("Test 3 (Invalid CSV):", res3.status_code, res3.json())

# Test Case 4: Oversized file (11 MB)
with open("large_file.txt", "rb") as f:
    res4 = client.post("/documents/upload", files={"file": ("large_file.txt", f, "text/plain")})
print("Test 4 (Large File):", res4.status_code, res4.json())

Test 1 (Valid PDF): 200 {'filename': 'employee_handbook.pdf', 'content_type': 'application/pdf', 'size_bytes': 1089, 'status': 'Uploaded successfully'}
Test 2 (Valid TXT): 200 {'filename': 'hr_policy.txt', 'content_type': 'text/plain', 'size_bytes': 241, 'status': 'Uploaded successfully'}
Test 3 (Invalid CSV): 400 {'detail': 'Invalid file type. Allowed types: .docx, .txt, .pdf'}
Test 4 (Large File): 400 {'detail': 'File size exceeds the 10 MB limit.'}


In [15]:
print("Saved files in folder:", os.listdir("uploaded_documents"))

Saved files in folder: ['hr_policy.txt', 'employee_handbook.pdf']


In [16]:
!pip install -q pymupdf

In [17]:
import fitz
import re

def extract_and_clean_text(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()
    raw_text = ""


    if ext == ".pdf":

        with fitz.open(file_path) as doc:
            for page_num, page in enumerate(doc):

                raw_text += f"\n--- Page {page_num + 1} ---\n"
                raw_text += page.get_text("text")

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

    # 2. Text Cleaning

    cleaned_text = re.sub(r'\n+', '\n', raw_text)
    cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text)


    cleaned_text = re.sub(r'--- Page \d+ ---', '', cleaned_text)

    return cleaned_text.strip()

# --- Practical Test
pdf_text = extract_and_clean_text("uploaded_documents/employee_handbook.pdf")
txt_text = extract_and_clean_text("uploaded_documents/hr_policy.txt")

print("=== Extracted PDF Text ===")
print(pdf_text)
print("\n=== Extracted TXT Text ===")
print(txt_text)

=== Extracted PDF Text ===
Employee Handbook - IT Security Guidelines
All employees must use VPN when accessing internal databases. Passwords must be rotated every
90 days.

=== Extracted TXT Text ===
COMPANY HR POLICY 2026
1. Annual Leave: Employees are allowed 20 paid leaves per calendar year.
2. Work From Home: Hybrid work allows up to 2 WFH days per week with manager approval.
3. Working Hours: Core hours are 10:00 AM to 6:00 PM IST.


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [18]:
@app.post("/documents/ingest")
async def ingest_document(file: UploadFile = File(...)):
    file_ext = os.path.splitext(file.filename)[1].lower()
    if file_ext not in ALLOWED_EXTENSIONS:
        raise HTTPException(status_code=400, detail="Invalid file type.")

    file_path = UPLOAD_DIR / file.filename
    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    await file.close()

    # Step 2 & 3: Extract & Clean Text
    extracted_text = extract_and_clean_text(str(file_path))

    return {
        "filename": file.filename,
        "total_characters": len(extracted_text),
        "extracted_preview": extracted_text[:150] + "...",
        "status": "Extracted and cleaned successfully"
    }

# Test
client = TestClient(app)
with open("employee_handbook.pdf", "rb") as f:
    res = client.post("/documents/ingest", files={"file": ("employee_handbook.pdf", f, "application/pdf")})
print(res.json())

{'filename': 'employee_handbook.pdf', 'total_characters': 145, 'extracted_preview': 'Employee Handbook - IT Security Guidelines\nAll employees must use VPN when accessing internal databases. Passwords must be rotated every\n90 days....', 'status': 'Extracted and cleaned successfully'}


In [19]:
!pip install -q langchain-text-splitters

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


sample_policy = """
1. Annual Leave Policy: All full-time employees are entitled to 20 paid annual leaves per calendar year. Unused leaves up to 10 days can be carried forward to the next year. Leave requests must be submitted on the HR portal at least 3 days in advance.

2. Work From Home (WFH) Policy: Hybrid work allows employees to work from home up to 2 days per week, subject to prior manager approval. During WFH, employees must remain available on Slack and Microsoft Teams during core working hours (10:00 AM to 6:00 PM IST).

3. IT Security & VPN Guidelines: All employees must connect to the company VPN when accessing internal databases, code repositories, or production servers. Passwords must be at least 12 characters long and rotated every 90 days. Sharing credentials is strictly prohibited.

4. Travel & Expense Reimbursement: Business travel expenses including flights, hotels, and meals will be reimbursed within 15 working days after submitting valid GST invoices on the finance portal.
"""

def create_chunks_with_metadata(text: str, filename: str, department: str):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=250,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    # Text ko chunks mein divide karein
    raw_chunks = text_splitter.split_text(text)


    structured_chunks = []
    for idx, chunk_text in enumerate(raw_chunks):
        structured_chunks.append({
            "chunk_id": f"{filename}_chunk_{idx + 1}",
            "text": chunk_text,
            "metadata": {
                "filename": filename,
                "department": department,
                "chunk_index": idx + 1,
                "total_chunks": len(raw_chunks)
            }
        })

    return structured_chunks

# Practical Run
chunks = create_chunks_with_metadata(
    text=sample_policy,
    filename="employee_handbook.pdf",
    department="HR"
)

print(f"Total Chunks Bane: {len(chunks)}\n")
for c in chunks:
    print(f"--- {c['chunk_id']} (Length: {len(c['text'])} chars) ---")
    print("Metadata:", c["metadata"])
    print("Content:", c["text"])
    print()

Total Chunks Bane: 7

--- employee_handbook.pdf_chunk_1 (Length: 172 chars) ---
Metadata: {'filename': 'employee_handbook.pdf', 'department': 'HR', 'chunk_index': 1, 'total_chunks': 7}
Content: 1. Annual Leave Policy: All full-time employees are entitled to 20 paid annual leaves per calendar year. Unused leaves up to 10 days can be carried forward to the next year

--- employee_handbook.pdf_chunk_2 (Length: 79 chars) ---
Metadata: {'filename': 'employee_handbook.pdf', 'department': 'HR', 'chunk_index': 2, 'total_chunks': 7}
Content: . Leave requests must be submitted on the HR portal at least 3 days in advance.

--- employee_handbook.pdf_chunk_3 (Length: 135 chars) ---
Metadata: {'filename': 'employee_handbook.pdf', 'department': 'HR', 'chunk_index': 3, 'total_chunks': 7}
Content: 2. Work From Home (WFH) Policy: Hybrid work allows employees to work from home up to 2 days per week, subject to prior manager approval

--- employee_handbook.pdf_chunk_4 (Length: 127 chars) ---
Metadata: {'f

In [21]:
@app.post("/documents/process")
async def process_document_pipeline(
    file: UploadFile = File(...),
    department: str = "General"
):
    # 1. Validate & Save
    file_ext = os.path.splitext(file.filename)[1].lower()
    if file_ext not in ALLOWED_EXTENSIONS:
        raise HTTPException(status_code=400, detail="Unsupported file format")

    file_path = UPLOAD_DIR / file.filename
    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    await file.close()

    # 2 & 3. Extract & Clean
    clean_text = extract_and_clean_text(str(file_path))

    # 4. Chunking with Metadata
    chunks_data = create_chunks_with_metadata(
        text=clean_text,
        filename=file.filename,
        department=department
    )

    return {
        "status": "Pipeline executed successfully",
        "filename": file.filename,
        "department": department,
        "total_chunks_created": len(chunks_data),
        "sample_chunk": chunks_data[0] if chunks_data else None
    }

# Test End-to-End Pipeline
client = TestClient(app)
with open("hr_policy.txt", "rb") as f:
    response = client.post(
        "/documents/process",
        params={"department": "HR"},
        files={"file": ("hr_policy.txt", f, "text/plain")}
    )

import json
print(json.dumps(response.json(), indent=2))

{
  "status": "Pipeline executed successfully",
  "filename": "hr_policy.txt",
  "department": "HR",
  "total_chunks_created": 1,
  "sample_chunk": {
    "chunk_id": "hr_policy.txt_chunk_1",
    "text": "COMPANY HR POLICY 2026\n1. Annual Leave: Employees are allowed 20 paid leaves per calendar year.\n2. Work From Home: Hybrid work allows up to 2 WFH days per week with manager approval.\n3. Working Hours: Core hours are 10:00 AM to 6:00 PM IST.",
    "metadata": {
      "filename": "hr_policy.txt",
      "department": "HR",
      "chunk_index": 1,
      "total_chunks": 1
    }
  }
}


In [22]:
!pip install -q sentence-transformers chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [23]:
from sentence_transformers import SentenceTransformer
import chromadb

# 1. Local Hugging Face Embedding Model load karein (384 dimensions)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. ChromaDB client aur collection initialize karein
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="company_policies")

# 3. Pichle step (Cell 9) ke chunks se texts, metadatas aur IDs alag karein
texts = [c["text"] for c in chunks]
metadatas = [c["metadata"] for c in chunks]
ids = [c["chunk_id"] for c in chunks]

# 4. Chunks ko numerical vectors (Embeddings) mein convert karein
embeddings = embedding_model.encode(texts).tolist()

print(f"Embedding vector ki length (dimensions): {len(embeddings[0])}")
print(f"Pehle chunk ke pehle 5 numbers: {embeddings[0][:5]}\n")

# 5. Vectors + Text + Metadata ko ChromaDB mein store karein
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=texts,
    metadatas=metadatas
)

print(f"Total {collection.count()} chunks ChromaDB mein successfully store ho gaye!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding vector ki length (dimensions): 384
Pehle chunk ke pehle 5 numbers: [0.055208828300237656, 0.0640607550740242, 0.046594809740781784, 0.0739191547036171, 0.07522699236869812]

Total 7 chunks ChromaDB mein successfully store ho gaye!


In [25]:
def search_documents(user_query: str, department_filter: str = None, top_k: int = 2):

    query_vector = embedding_model.encode([user_query]).tolist()


    where_clause = {"department": department_filter} if department_filter else None


    results = collection.query(
        query_embeddings=query_vector,
        n_results=top_k,
        where=where_clause
    )
    return results

# Test Query: Document mein "paid annual leaves" likha hai, hum "chutti" / "vacation days" poochte hain
query = "How many vacation days do employees get in a year?"
search_results = search_documents(query, department_filter="HR", top_k=2)

print(f"User Query: '{query}'\n")
for i in range(len(search_results["documents"][0])):
    print(f"Match {i+1} (Distance: {search_results['distances'][0][i]:.4f}):")
    print("Metadata:", search_results["metadatas"][0][i])
    print("Matched Chunk:", search_results["documents"][0][i])
    print("-" * 50)

User Query: 'How many vacation days do employees get in a year?'

Match 1 (Distance: 1.3529):
Metadata: {'chunk_index': 3, 'total_chunks': 7, 'department': 'HR', 'filename': 'employee_handbook.pdf'}
Matched Chunk: 2. Work From Home (WFH) Policy: Hybrid work allows employees to work from home up to 2 days per week, subject to prior manager approval
--------------------------------------------------
Match 2 (Distance: 1.3663):
Metadata: {'department': 'HR', 'chunk_index': 5, 'filename': 'employee_handbook.pdf', 'total_chunks': 7}
Matched Chunk: 3. IT Security & VPN Guidelines: All employees must connect to the company VPN when accessing internal databases, code repositories, or production servers. Passwords must be at least 12 characters long and rotated every 90 days
--------------------------------------------------


In [26]:
@app.post("/documents/ingest-rag")
async def ingest_to_vector_db(file: UploadFile = File(...), department: str = "HR"):
    # 1. Save file
    file_path = UPLOAD_DIR / file.filename
    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
    await file.close()

    # 2. Extract & Clean
    clean_text = extract_and_clean_text(str(file_path))

    # 3. Chunk + Metadata
    chunks_data = create_chunks_with_metadata(clean_text, file.filename, department)

    # 4. Embed & Store in ChromaDB
    texts = [c["text"] for c in chunks_data]
    metadatas = [c["metadata"] for c in chunks_data]
    ids = [c["chunk_id"] for c in chunks_data]
    vectors = embedding_model.encode(texts).tolist()

    collection.upsert(ids=ids, embeddings=vectors, documents=texts, metadatas=metadatas)

    return {
        "status": "Stored in Vector DB",
        "filename": file.filename,
        "chunks_indexed": len(chunks_data)
    }

@app.get("/documents/search")
async def query_knowledge_base(q: str, department: str = "HR"):
    results = search_documents(q, department_filter=department, top_k=2)
    return {
        "query": q,
        "retrieved_chunks": results["documents"][0],
        "metadata": results["metadatas"][0]
    }

# Test Full Pipeline
client = TestClient(app)
with open("employee_handbook.pdf", "rb") as f:
    print("Ingest:", client.post("/documents/ingest-rag", params={"department": "IT"}, files={"file": ("employee_handbook.pdf", f, "application/pdf")}).json())

print("Search:", client.get("/documents/search", params={"q": "What are the VPN and password rules?", "department": "IT"}).json())

Ingest: {'status': 'Stored in Vector DB', 'filename': 'employee_handbook.pdf', 'chunks_indexed': 1}
Search: {'query': 'What are the VPN and password rules?', 'retrieved_chunks': ['Employee Handbook - IT Security Guidelines\nAll employees must use VPN when accessing internal databases. Passwords must be rotated every\n90 days.'], 'metadata': [{'chunk_index': 1, 'filename': 'employee_handbook.pdf', 'department': 'IT', 'total_chunks': 1}]}


In [27]:
import json
from fastapi.testclient import TestClient

client = TestClient(app)

print("=== TEST 1: Finance Department ki nayi policy upload & index karna ===")

with open("finance_policy.txt", "w") as f:
    f.write("""
    FINANCE & REIMBURSEMENT POLICY 2026:
    1. Employees can claim up to Rs. 5,000 per month for mobile and broadband internet bills.
    2. Flight bookings for client visits must be approved by the VP of Finance at least 7 days before travel.
    3. Expense receipts must be uploaded on the Zoho Expense portal within 10 days of return.
    """)

with open("finance_policy.txt", "rb") as f:
    ingest_res = client.post(
        "/documents/ingest-rag",
        params={"department": "Finance"},
        files={"file": ("finance_policy.txt", f, "text/plain")}
    )
print("Ingest Response:", json.dumps(ingest_res.json(), indent=2))


print("\n=== TEST 2: Semantic Search (Synonyms ke saath) ===")

search_res_1 = client.get(
    "/documents/search",
    params={
        "q": "How much money can I claim for wifi recharge?",
        "department": "Finance"
    }
)
print("Search Result (Valid Department):", json.dumps(search_res_1.json(), indent=2))


print("\n=== TEST 3: Security & Metadata Filter Check (Wrong Department) ===")

search_res_2 = client.get(
    "/documents/search",
    params={
        "q": "What is the limit for internet bill reimbursement?",
        "department": "HR"  # Galat department filter
    }
)
print("Search Result (HR trying to access Finance):", json.dumps(search_res_2.json(), indent=2))

=== TEST 1: Finance Department ki nayi policy upload & index karna ===
Ingest Response: {
  "status": "Stored in Vector DB",
  "filename": "finance_policy.txt",
  "chunks_indexed": 2
}

=== TEST 2: Semantic Search (Synonyms ke saath) ===
Search Result (Valid Department): {
  "query": "How much money can I claim for wifi recharge?",
  "retrieved_chunks": [
    "FINANCE & REIMBURSEMENT POLICY 2026:\n 1. Employees can claim up to Rs. 5,000 per month for mobile and broadband internet bills.\n 2. Flight bookings for client visits must be approved by the VP of Finance at least 7 days before travel.",
    "3. Expense receipts must be uploaded on the Zoho Expense portal within 10 days of return."
  ],
  "metadata": [
    {
      "filename": "finance_policy.txt",
      "total_chunks": 2,
      "chunk_index": 1,
      "department": "Finance"
    },
    {
      "filename": "finance_policy.txt",
      "department": "Finance",
      "total_chunks": 2,
      "chunk_index": 2
    }
  ]
}

=== TEST 3:

In [28]:
!pip install -q pyngrok nest-asyncio uvicorn

In [30]:

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import uvicorn
import threading
import subprocess
import time
import re


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)


tunnel = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print(f"🚀 Swagger UI kholne ke liye is link par click karein: {match.group(0)}/docs")
        break

INFO:     Started server process [3100]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 Swagger UI kholne ke liye is link par click karein: https://theater-suse-housing-ppm.trycloudflare.com/docs
